# Оценка F1-fast: Qwen + LoRA-адаптер

Этот ноутбук сравнивает обученную F1-fast с исходной B0 на **одних и тех же** 100 заданиях GQA-ru и 100 заданиях MMBench-ru. Список идентификаторов берётся из сохранённых предсказаний baseline, поэтому случайная повторная выборка исключена.

LoRA-адаптер содержит только выученные изменения весов (~74 МБ). Он подключается поверх исходной Qwen2.5-VL-3B; вместе они образуют F1-fast.

In [1]:
from pathlib import Path
import re
import time

import pandas as pd
import torch
from datasets import load_dataset
from peft import PeftModel
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BASELINE_PATH = ROOT / 'results' / 'baseline_predictions.csv'
ADAPTER_DIR = ROOT / 'models' / 'f1_fast_qlora_adapter'
RESULTS_DIR = ROOT / 'results'
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
assert ADAPTER_DIR.exists(), 'Не найден адаптер F1.'
assert torch.cuda.is_available(), 'Для оценки нужна CUDA.'
baseline = pd.read_csv(BASELINE_PATH)
print('GPU:', torch.cuda.get_device_name(0))
display(baseline.groupby('benchmark')['sample_id'].count())

GPU: NVIDIA GeForce RTX 4060


benchmark
GQA-ru        100
MMBench-ru    100
Name: sample_id, dtype: int64

## 1. Восстановление фиксированного набора оценки

In [2]:
gqa_baseline = baseline.query("benchmark == 'GQA-ru'").copy()
mmbench_baseline = baseline.query("benchmark == 'MMBench-ru'").copy()

gqa_questions = load_dataset('deepvk/GQA-ru', 'testdev_balanced_instructions', split='testdev')
gqa_images = load_dataset('deepvk/GQA-ru', 'testdev_balanced_images', split='testdev')
image_by_id = {str(row['id']): row['image'] for row in gqa_images}
gqa_by_id = {str(row['id']): row for row in gqa_questions}
gqa_eval = []
for sample_id in gqa_baseline['sample_id'].astype(str):
    row = gqa_by_id[sample_id]
    gqa_eval.append({'id': sample_id, 'image': image_by_id[str(row['imageId'])], 'question': row['question'], 'answer': row['answer']})

mmbench = load_dataset('deepvk/MMBench-ru', split='dev')
mmbench_by_id = {str(row['index']): row for row in mmbench}
mmbench_eval = [mmbench_by_id[str(sample_id)] for sample_id in mmbench_baseline['sample_id']]

assert len(gqa_eval) == len(gqa_baseline) == 100
assert len(mmbench_eval) == len(mmbench_baseline) == 100
print('GQA-ru:', len(gqa_eval), '| MMBench-ru:', len(mmbench_eval))

GQA-ru: 100 | MMBench-ru: 100


## 2. Загрузка Qwen и адаптера F1

In [3]:
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quantization, device_map='auto', torch_dtype=torch.float16
).eval()
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()
print('F1 загружена. VRAM, МБ:', round(torch.cuda.memory_allocated() / 1024**2))

W0714 23:16:46.047000 15292 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

F1 загружена. VRAM, МБ: 2535


In [4]:
def generate_answer(image, prompt, max_new_tokens):
    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(output[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

def normalize_gqa(text):
    return re.sub(r'[^а-яёa-z0-9-]', '', text.lower())

def extract_option(text):
    match = re.search(r'(?<![A-ZА-Я])[ABCD](?![A-ZА-Я])', text.upper())
    return match.group(0) if match else ''

## 3. Оценка F1 на тех же промптах и метриках

In [5]:
gqa_rows = []
started = time.perf_counter()
for i, row in enumerate(gqa_eval, start=1):
    prompt = f"Вопрос: {row['question']}\nОтветь одним словом на русском языке, без пояснений."
    prediction = generate_answer(row['image'], prompt, max_new_tokens=12)
    gqa_rows.append({'benchmark': 'GQA-ru', 'sample_id': row['id'], 'question': row['question'], 'target': row['answer'], 'prediction': prediction, 'is_correct': normalize_gqa(prediction) == normalize_gqa(row['answer'])})
    if i % 10 == 0: print(f'GQA-ru: {i}/{len(gqa_eval)}')
gqa_results = pd.DataFrame(gqa_rows)
print(f"GQA-ru Exact Match: {gqa_results.is_correct.mean():.1%}; {(time.perf_counter()-started)/60:.1f} мин.")

c:\Users\miste\miniconda3\Lib\site-packages\transformers\tokenization_utils_base.py:2368: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


c:\Users\miste\miniconda3\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


GQA-ru: 10/100


GQA-ru: 20/100


GQA-ru: 30/100


...

GQA-ru: 60/100


GQA-ru: 70/100


GQA-ru: 80/100


GQA-ru: 90/100


GQA-ru: 100/100
GQA-ru Exact Match: 43.0%; 1.1 мин.


In [6]:
mmbench_rows = []
started = time.perf_counter()
for i, row in enumerate(mmbench_eval, start=1):
    options = '\n'.join(f'{letter}. {row[letter]}' for letter in 'ABCD')
    prompt = f"Вопрос: {row['question']}\n{options}\n\nВыбери правильный вариант. Ответь только одной латинской буквой: A, B, C или D."
    prediction = generate_answer(row['image'], prompt, max_new_tokens=8)
    option = extract_option(prediction)
    mmbench_rows.append({'benchmark': 'MMBench-ru', 'sample_id': row['index'], 'question': row['question'], 'target': row['answer'], 'prediction': prediction, 'selected_option': option, 'is_correct': option == row['answer']})
    if i % 10 == 0: print(f'MMBench-ru: {i}/{len(mmbench_eval)}')
mmbench_results = pd.DataFrame(mmbench_rows)
print(f"MMBench-ru Accuracy: {mmbench_results.is_correct.mean():.1%}; {(time.perf_counter()-started)/60:.1f} мин.")

MMBench-ru: 10/100


MMBench-ru: 20/100


MMBench-ru: 30/100


MMBench-ru: 40/100


MMBench-ru: 50/100


MMBench-ru: 60/100


MMBench-ru: 70/100


MMBench-ru: 80/100


MMBench-ru: 90/100


MMBench-ru: 100/100
MMBench-ru Accuracy: 74.0%; 0.7 мин.


In [7]:
f1_results = pd.concat([gqa_results, mmbench_results], ignore_index=True)
f1_results.to_csv(RESULTS_DIR / 'f1_fast_predictions.csv', index=False)
f1_metrics = (f1_results.groupby('benchmark')['is_correct'].mean().mul(100).round(1).rename('F1-fast'))
b0_metrics = (baseline.groupby('benchmark')['is_correct'].mean().mul(100).round(1).rename('B0'))
comparison = pd.concat([b0_metrics, f1_metrics], axis=1)
comparison['change_pp'] = (comparison['F1-fast'] - comparison['B0']).round(1)
comparison.to_csv(RESULTS_DIR / 'f1_fast_metrics.csv')
display(comparison)

errors = f1_results.loc[~f1_results.is_correct, ['benchmark', 'question', 'target', 'prediction']]
display(errors.head(15))
print('Результаты сохранены в results/f1_fast_predictions.csv и results/f1_fast_metrics.csv')

,B0,F1-fast,change_pp
benchmark,,,
GQA-ru,43.0,43.0,0.0
MMBench-ru,74.0,74.0,0.0


,benchmark,question,target,prediction
5,GQA-ru,"Как называется предмет мебели, стоящий на ковре?",стол,Стул
7,GQA-ru,Какая еда не жёлтая?,пепперони,Пицца
8,GQA-ru,Какой вид транспортного средства припаркован?,автомобили,Автомобиль
11,GQA-ru,Что находится на полу?,диван,Ковер
13,GQA-ru,Какое устройство находится слева от стула?,роутер,Компьютер
14,GQA-ru,Кто носит обувь?,футболист,Женщины
16,GQA-ru,Что стоит на полке?,украшение,Свечи
17,GQA-ru,Кто носит очки?,джентльмен,Пожилой мужчина.
19,GQA-ru,Что стоит рядом с цветами?,стул,Столик.
22,GQA-ru,Брокколи в чём?,корзине,На рынке.


Результаты сохранены в results/f1_fast_predictions.csv и results/f1_fast_metrics.csv
